### Summary
- **Model type:** Linear programming (continuous decision variables)
- **Objective:** Maximize total trading profit
- **Constraints:** Respect supply and market constraints
- **Solver:** HiGHS via Pyomo framework

## Problem

An energy trading company purchases electricity from two different sources and sells it in the wholesale electricity market. The company aims to determine the optimal amount of electricity to purchase from each source to maximize its total trading profit while respecting supply and market constraints.

The company can purchase electricity from:

- **Source A:** Electricity purchased at a cost of **£40** per MWh and sold at a market price of **£70** per MWh.
- **Source B:** Electricity purchased at a cost of **£50** per MWh and sold at a market price of **£70** per MWh.

The company can purchase a **maximum** of **100 MWh** from **Source A** and **80 MWh** from **Source B**. Due to market demand and trading commitments, the <u>total amount of electricity sold cannot exceed 150 MWh</u>.

The objective is to determine how much electricity to purchase from each source to maximize total profit.

## Mathematical Model

Maximize total trading profit:

\begin{align}
    \text{max}~~  & 30x + 20y \\
    \text{s.t.}~~ & x \le 100 \\
                  & y \le 80 \\
                  & x + y \le 150 \\
                  & x, y \ge 0
\end{align}


<div style="border-radius:10px; border: #0b0b0b solid; padding: 15px; background-color: #2b2e2f; font-size:100%; text-align:left">
<h3 align="left"><font color='#c8c8d0'>❕Solution Approach</font></h3> 
<p style="color: #9e9aa2;">The example assumes that all purchased electricity can be sold at a fixed market price of £70 per MWh, with no transmission costs, trading fees or price uncertainty.</p>



In [1]:
# Import libraries
import pyomo.environ as pe
from pyomo.opt import TerminationCondition

In [5]:
# Pyomo Model
model = pe.ConcreteModel()

In [6]:
# Decision Variables (MWh)
model.source_A = pe.Var(domain=pe.NonNegativeReals)
model.source_B = pe.Var(domain=pe.NonNegativeReals)

In [ ]:
# Objective Fct
model.profit = pe.Objective(
    expr=30 * model.source_A + 20 * model.source_B,
    sense=pe.maximize
)

In [8]:
# Constraints
model.source_A_constraint = pe.Constraint(
    expr=model.source_A <= 100
)

model.source_B_constraint = pe.Constraint(
    expr=model.source_B <= 80
)

model.market_constraint = pe.Constraint(
    expr=model.source_A + model.source_B <= 150
)

In [ ]:
# Solver
solver = pe.SolverFactory("appsi_highs")
if not solver.available(exception_flag=False):
    raise RuntimeError(
        "Solver '{solver_name}' is unavailable."
    )

In [ ]:
# Solve Optim Model
results = solver.solve(model, tee=True)

In [ ]:
# Display Results
if results.solver.termination_condition == TerminationCondition.optimal:

    source_A = pe.value(model.source_A)
    source_B = pe.value(model.source_B)
    profit = pe.value(model.profit)

    print("========== OPTIMAL SOLUTION ==========")
    print(f"Electricity from Source A: {source_A:.2f} MWh")
    print(f"Electricity from Source B: {source_B:.2f} MWh")
    print(f"Maximum trading profit: £{profit:.2f}")

    print("\n========== RESOURCE UTILIZATION ==========")
    print(f"Source A used: {source_A:.2f} / 100 MWh")
    print(f"Source B used: {source_B:.2f} / 80 MWh")
    print(
        f"Market capacity used: "
        f"{source_A + source_B:.2f} / 150 MWh"
    )
    print(
        f"Unused market capacity: "
        f"{150 - source_A - source_B:.2f} MWh"
    )

else:
    print("No optimal solution found.")
    print("Solver status:", results.solver.status)

========== OPTIMAL SOLUTION ==========
Electricity from Source A: 100.00 MWh
Electricity from Source B: 50.00 MWh
Maximum trading profit: £4000.00

========== RESOURCE UTILIZATION ==========
Source A used: 100.00 / 100 MWh
Source B used: 50.00 / 80 MWh
Market capacity used: 150.00 / 150 MWh
Unused market capacity: 0.00 MWh
